In [13]:
from dotenv import load_dotenv
load_dotenv()
import os

In [14]:
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth
from datetime import datetime
import time
import json
from urllib.parse import urlencode

In [15]:
# ---------- SETTINGS ----------

username = os.getenv("USERNAME")
password = os.getenv("PASSWORD")

base_url = "https://eu1.eam.hxgnsmartcloud.com:443/axis/restservices/workorders/"
headers = {
    "Content-Type": "application/json",
    "tenant": "WEPADE_TST",
    "organization": "MUE",
}


def parse_date_field(date_obj):
    if not date_obj or not isinstance(date_obj, dict):
        return None

    try:
        # Extract the actual year 
        raw_ts = date_obj.get("YEAR")
        if isinstance(raw_ts, (int, float)):
            # Convert to datetime
            dt_from_ts = datetime.fromtimestamp(raw_ts / 1000)
            actual_year = dt_from_ts.year
        else:
            actual_year = datetime.now().year  # fallback

        month = int(date_obj.get("MONTH", 1))
        day = int(date_obj.get("DAY", 1))
        hour = int(date_obj.get("HOUR", 0))
        minute = int(date_obj.get("MINUTE", 0))
        second = int(date_obj.get("SECOND", 0))

        dt = datetime(actual_year, month, day, hour, minute, second)
        return dt.strftime("%d-%b-%Y %H:%M")

    except Exception as e:
        print(f"Failed to parse datetime: {e} | Raw: {date_obj}")
        return None

def get_nested(d, *keys):
    for key in keys:
        if isinstance(d, dict):
            d = d.get(key)
        else:
            return None
    return d

def extract_fields(workorder):
    return {
        "workorder_ID": get_nested(workorder, "WORKORDERID", "JOBNUM"),
        "workorder_description": get_nested(workorder, "WORKORDERID", "DESCRIPTION"),
        "workorder_type": get_nested(workorder, "TYPE", "DESCRIPTION"),
        "asset_ID": get_nested(workorder, "EQUIPMENTID", "EQUIPMENTCODE"),
        "workorder_status": get_nested(workorder, "STATUS", "DESCRIPTION"),
        "safety": workorder.get("SAFETY"),
        "department": get_nested(workorder, "DEPARTMENTID", "DESCRIPTION"),
        "remark_available": None,
        "parts": None,
        "hired_labour": None,
        "created_by": get_nested(workorder, "CREATEDBY", "DESCRIPTION") or workorder.get("ENTEREDBY"),
        "date_time_created": parse_date_field(workorder.get("CREATEDDATE")),
        "failure_code": workorder.get("FAILURECODEID"),
        "action_code": workorder.get("ACTIONCODEID"),
        "cause_code": workorder.get("CAUSECODEID"),
        "priority": get_nested(workorder, "PRIORITY", "DESCRIPTION"),
        "cost_code": get_nested(workorder, "COSTCODEID", "COSTCODE"),
        "original_pm_due_date": None,
        "belongs_to": None,
        "near_miss": None,
        "reporter": get_nested(workorder, "REQUESTEDBY", "DESCRIPTION"),
        "date_reported": parse_date_field(workorder.get("REPORTED")),
        "assigned_by": workorder.get("ASSIGNEDBYNAME"),
        "assigned_to": workorder.get("ASSIGNEDTO"),
        "scheduled_start_date": parse_date_field(workorder.get("TARGETDATE")),
        "scheduled_end_date": parse_date_field(workorder.get("SCHEDEND")),
        "downtime_hours": workorder.get("DOWNTIMEHOURS")
    }

In [16]:
def fetch_all_summary_records(max_records):
    all_records = []
    cursor = 0
    page_size = 50  # fixed page size in EAM

    while True:
        # Set cursorposition in headers
        page_headers = headers.copy()
        page_headers["cursorposition"] = str(cursor)  # must be a string

        # Make GET request
        res = requests.get(base_url.rstrip('/'), headers=page_headers, auth=HTTPBasicAuth(username, password))

        if res.status_code != 200:
            print(f"Failed to fetch page at cursor {cursor}: {res.status_code}")
            break

        data = res.json()
        result_data = data.get("Result", {}).get("ResultData", {})
        records = result_data.get("DATARECORD", [])

        if not records:
            print("No more records found.")
            break

        all_records.extend(records)
        print(f"Got {len(records)} work orders (total so far: {len(all_records)})")

        if len(all_records) >= max_records:
            print(f"Reached max limit of {max_records}")
            all_records = all_records[:max_records]
            break

        # Advance the cursor to the next page
        cursor += page_size

    return all_records


In [ ]:
def fetch_and_save_detailed_workorders():
    record_list = fetch_all_summary_records(max_records=150)
    print(f"Found {len(record_list)} work orders in summary list...")

    # Load configurable values from .env
    org_filter = os.getenv("ORG_FILTER", "MUE")
    output_folder = os.getenv("OUTPUT_PATH", "./output")
    output_filename = os.getenv("OUTPUT_FILE", "workorders.xlsx")

    all_details = []

    for record in record_list:
        jobnum = get_nested(record, "WORKORDERID", "JOBNUM")
        orgcode = get_nested(record, "WORKORDERID", "ORGANIZATIONID", "ORGANIZATIONCODE")

        if not jobnum or not orgcode:
            print("⚠️ Missing JOBNUM or ORGCODE in record, skipping...")
            continue

        if orgcode != org_filter:
            continue

        full_id = f"{jobnum}%23{orgcode}"  # encode "#" as %23
        url = base_url + full_id

        try:
            res = requests.get(url, headers=headers, auth=HTTPBasicAuth(username, password))
            if res.status_code == 200:
                full_data = res.json()
                wo_detail = full_data.get("Result", {}).get("ResultData", {}).get("WorkOrder")
                if wo_detail:
                    extracted = extract_fields(wo_detail)
                    all_details.append(extracted)
            else:
                print(f"Skipped {full_id} due to status {res.status_code}")
        except Exception as e:
            print(f"Error fetching {full_id}: {e}")

        time.sleep(0.2)

    # Save to Excel
    if all_details:
        df = pd.DataFrame(all_details)
        os.makedirs(output_folder, exist_ok=True)
        output_file = os.path.join(output_folder, output_filename)
        df.to_excel(output_file, index=False)
        print(f"Saved detailed work orders to:\n{output_file}")
    else:
        print("No detailed work orders extracted.")


In [18]:
def fetch_all_workorders_data():
    try:
        response = requests.get(base_url, headers=headers, auth=HTTPBasicAuth(username, password))
        if response.status_code == 200:
            print("Summary list fetched successfully.")
            return response.json()
        else:
            print(f"Failed to fetch summary list: {response.status_code}")
            return {}
    except Exception as e:
        print(f"Error fetching summary list: {e}")
        return {}

latest_data = fetch_all_workorders_data()

fetch_and_save_detailed_workorders()

Summary list fetched successfully.
Got 50 work orders (total so far: 50)
Got 50 work orders (total so far: 100)
Got 50 work orders (total so far: 150)
Reached max limit of 150
Found 150 work orders in summary list...
Saved detailed work orders to:
D:\WEPA\EAM\EAM1\ALL_WO\TRUE\workorders_1002.xlsx
